In [2]:
from ase.build import bulk
from ase.visualize import view
from ase.visualize.plot import plot_atoms
from gpaw import GPAW, PW
from ase.dft.dos import DOS



import nglview as ngv
import matplotlib.pyplot as plt


# Create silicon crystal (diamond cubic structure)
si = bulk('Si', 'diamond', a=5.43)

# Repeat to make it bigger (more visible)
si = si.repeat((2, 2, 2))
si.set_cell(si.cell * 1, scale_atoms=True)

# Visualize
#view(si)

view = ngv.show_ase(si)
view
#plot_atoms(si)
#plt.savefig("structure.png")

NGLWidget()

In [3]:
# Bulk DFT calculation

# calculator
calc = GPAW(
    mode=PW(350),
    kpts=[4, 4, 4], 
    txt='gpaw.bulk_Si.txt'
    #setups={'Si': '11'}
)

si.calc = calc
print(
    'Bulk {0} potential energy = {1:.3f}eV'.format(
        si.get_chemical_formula(), si.get_potential_energy()
    )
)

Bulk Si16 potential energy = -95.072eV


In [4]:
ground_state_file = 'bulk_Si_groundstate.gpw'
calc.write(ground_state_file)

In [5]:
calc = GPAW(ground_state_file)
dos = DOS(calc, npts=800, width=0)
energies = dos.get_energies()
weights = dos.get_dos()

In [6]:
fig, ax = plt.subplots()
ax.axvline(0.0, linestyle='--', color='black', alpha=0.5)
ax.plot(energies, weights)
ax.set_xlabel('Energy - Fermi Energy (eV)')
ax.set_ylabel('Density of States (1/eV)')
fig.tight_layout()

In [7]:
calc = GPAW(ground_state_file)
dos = DOS(calc, npts=800, width=0)
energies = dos.get_energies()
weights = dos.get_dos()

In [8]:
lat = si.cell.get_bravais_lattice()
print(lat.description())

FCC(a=10.86)
  Variant name: FCC
  Special point names: GKLUWX
  Default path: GXWKGLUWLK,UX

  Special point coordinates:
    G   0.0000  0.0000  0.0000
    K   0.3750  0.3750  0.7500
    L   0.5000  0.5000  0.5000
    U   0.6250  0.2500  0.6250
    W   0.5000  0.2500  0.7500
    X   0.5000  0.0000  0.5000



In [9]:
lat.plot_bz(show=True)
plt.show()

/mnt/c/UNIVIE/Piezoresistivity-simulations/wsl_venv/lib/python3.12/site-packages/ase/dft/bz.py:362: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_34001/1329312272.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
path = si.cell.bandpath('WLGXWK', density=10)
path.write('path.json')
print(path)

BandPath(path='WLGXWK', cell=[3x3], special_points={GKLUWX}, kpts=[20x3])


In [11]:
path.plot()
plt.show()

/tmp/ipykernel_34001/648803344.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
calc = GPAW(ground_state_file)
calc = calc.fixed_density(kpts=path, symmetry='off')


  ___ ___ ___ _ _ _  
 |   |   |_  | | | | 
 | | | | | . | | | | 
 |__ |  _|___|_____|  25.7.0
 |___|_|             

User:   ysafi@YusufSafi
Date:   Tue May 12 11:47:41 2026
Arch:   x86_64
Pid:    34001
CWD:    /mnt/c/UNIVIE/Piezoresistivity-simulations/AEStructure
Python: 3.12.3
gpaw:   /mnt/c/UNIVIE/Piezoresistivity-simulations/wsl_venv/lib/python3.12/site-packages/gpaw
_gpaw:  /mnt/c/UNIVIE/Piezoresistivity-simulations/wsl_venv/lib/python3.12/site-packages/
        _gpaw.cpython-312-x86_64-linux-gnu.so
ase:    /mnt/c/UNIVIE/Piezoresistivity-simulations/wsl_venv/lib/python3.12/site-packages/ase (version 3.28.0)
numpy:  /mnt/c/UNIVIE/Piezoresistivity-simulations/wsl_venv/lib/python3.12/site-packages/numpy (version 2.4.4)
scipy:  /mnt/c/UNIVIE/Piezoresistivity-simulations/wsl_venv/lib/python3.12/site-packages/scipy (version 1.17.1)
libxc:  2.x.y
units:  Angstrom and eV
cores: 1
OpenMP: False
OMP_NUM_THREADS: 1

Input parameters:
  gpts: [35 35 35]
  kpts: {cell: Cell([[0.0, 5.4300000

In [13]:
bs = calc.band_structure()
bs.write('bs.json')
print(bs)

BandStructure(path=BandPath(path='WLGXWK', cell=[3x3], special_points={GKLUWX}, kpts=[20x3]), energies=[1x20x43 values], reference=5.374868676236262)


In [14]:
ax = bs.plot()
ax.set_ylim(-2.0, 30.0)
plt.show()

/tmp/ipykernel_34001/2336608413.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Thesis Figure — Band Structure + Deformation Potential Schematic

This cell produces **Figure 6** of the thesis (Section 8).

**Left panel:** Silicon band structure.
- If `bs.json` exists (i.e. the GPAW band-structure run above has completed),
  it is loaded directly and plotted — this gives the actual DFT result.
- If `bs.json` is not yet available, an analytic reconstruction is used
  (empirical pseudopotential features: indirect gap 1.12 eV at Δ, three
  valence bands, two conduction bands along Γ→X→K→Γ→L).

**Right panel:** Schematic of the hydrostatic-strain-induced band-edge shift
from deformation potential theory (Hamaguchi §4.7.2):
$$\Delta E_g = (a_c + a_v)\,\varepsilon_h$$
showing how tensile hydrostatic strain widens the gap and hence reduces the
intrinsic carrier density and conductivity.

In [16]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Try to load the real GPAW band structure ──────────────────────────────
BS_FILE = 'bs.json'
USE_GPAW = os.path.exists(BS_FILE)

C_VB    = 'tab:blue'
C_CB    = 'tab:red'
C_GREY  = 'dimgray'
C_GREEN = 'forestgreen'

fig, (ax_bs, ax_sc) = plt.subplots(
    1, 2, figsize=(13, 5.8),
    gridspec_kw={'width_ratios': [1.4, 1]}
)

# ════════════════════════════════════════════════════════════════════════════
# LEFT PANEL — band structure
# ════════════════════════════════════════════════════════════════════════════
if USE_GPAW:
    # ── Real GPAW output ──────────────────────────────────────────────────
    from ase.spectrum.band_structure import BandStructure
    bs_gpaw = BandStructure.read(BS_FILE)

    # bs.plot() returns an Axes; re-use our ax_bs
    bs_gpaw.plot(ax=ax_bs, emin=-7.5, emax=7.5, show=False)

    # Restyle to match thesis palette
    for line in ax_bs.get_lines():
        e = line.get_ydata()
        if e is not None and len(e):
            col = C_CB if np.min(e) > 0.5 else C_VB
            line.set_color(col)
            line.set_linewidth(1.6)

    ax_bs.set_title(
        'Silicon band structure (GPAW / DFT)\n'
        r'($\Gamma \to X \to K \to \Gamma \to L$)', fontsize=11)
    print('Loaded real GPAW band structure from', BS_FILE)

else:
    # ── Analytic fallback (empirical pseudopotential features) ────────────
    print('bs.json not found — using analytic reconstruction.')

    SEG  = np.array([1.000, 0.354, 1.061, 0.866])
    KPTS = np.concatenate([[0], np.cumsum(SEG)])
    KLBLS = [r'$\Gamma$', 'X', 'K', r'$\Gamma$', 'L']
    N = 150

    k = np.concatenate([
        np.linspace(KPTS[i], KPTS[i+1], N, endpoint=(i == 3))
        for i in range(4)
    ])

    def t(i): return np.linspace(0, 1, N, endpoint=(i == 3))
    t0, t1, t2, t3 = t(0), t(1), t(2), t(3)

    # Valence bands
    vb4 = np.concatenate([
        -12.5*t0**2 + 0.15*np.sin(np.pi*t0),
        -12.5*t0[-1] - 1.8*(t1-0.5)**2 + 0.45,
        np.linspace(-12.5*t0[-1] - 1.8*(t1[-1]-0.5)**2 + 0.45, 0.0, N),
        -1.0 - 5.5*t3**2,
    ])
    vb3 = np.concatenate([
        -0.04 - 13.0*t0**2 - 0.4*np.sin(np.pi*t0),
        -12.5*t0[-1] - 1.8*(t1-0.5)**2 + 0.30,
        np.linspace(-12.5*t0[-1] - 1.8*(t1[-1]-0.5)**2 + 0.30, -0.04, N),
        -1.05 - 5.7*t3**2,
    ])
    vb2 = np.concatenate([
        -2.9 - 5.5*t0**2 + 2.0*t0,
        np.linspace(-2.9 - 5.5 + 2.0, -5.8, N),
        np.linspace(-5.8, -2.9, N),
        -2.9 - 0.8*t3,
    ])

    # Conduction bands
    cb5 = np.concatenate([
        8.5*t0**2 - 9.0*t0 + 3.8,          # min ~1.12 eV at t ≈ 0.85
        np.linspace(8.5 - 9.0 + 3.8, 3.5, N),
        np.linspace(3.5, 3.8, N),
        2.1 + 1.0*t3 + 0.5*t3**2,
    ])
    cb6 = np.concatenate([
        4.2 - 2.5*t0 + 3.5*t0**2,
        np.linspace(4.2 - 2.5 + 3.5, 5.2, N),
        np.linspace(5.2, 4.0, N),
        4.0 + 2.0*t3**2,
    ])

    for band, col in [(vb4, C_VB), (vb3, C_VB), (vb2, C_VB),
                      (cb5, C_CB), (cb6, C_CB)]:
        ax_bs.plot(k, band, color=col, lw=1.8)

    # VBM reference
    ax_bs.axhline(0, color='k', lw=0.7, ls='--', alpha=0.5)

    # Band gap annotation
    cbm_idx = np.argmin(cb5[:N])
    cbm_k, cbm_e = k[cbm_idx], cb5[cbm_idx]
    ax_bs.annotate('', xy=(0.02, cbm_e), xytext=(0.02, 0.0),
                   arrowprops=dict(arrowstyle='<->', color=C_GREEN, lw=1.6))
    ax_bs.text(0.08, cbm_e/2,
               rf'$E_g = {cbm_e:.2f}\,\mathrm{{eV}}$',
               color=C_GREEN, fontsize=9, va='center')
    ax_bs.plot(cbm_k, cbm_e, 'o', color=C_CB, ms=5.5, zorder=5)
    ax_bs.text(cbm_k+0.05, cbm_e+0.08, r'CBM ($\Delta$)',
               color=C_CB, fontsize=8)

    for xk, lbl in zip(KPTS, KLBLS):
        ax_bs.axvline(xk, color='gray', lw=0.6, alpha=0.4)
        ax_bs.text(xk, -7.85, lbl, ha='center', fontsize=11)

    ax_bs.set_xlim(k[0], k[-1])
    ax_bs.set_ylim(-7.5, 7.5)
    ax_bs.set_xticks([])
    ax_bs.set_ylabel('Energy (eV)', fontsize=11)
    ax_bs.set_title(
        'Silicon band structure (analytic reconstruction)\n'
        r'($\Gamma \to X \to K \to \Gamma \to L$)', fontsize=11)

ax_bs.legend(
    handles=[mpatches.Patch(color=C_VB, label='Valence bands'),
             mpatches.Patch(color=C_CB, label='Conduction bands')],
    fontsize=9, loc='upper right'
)

# ════════════════════════════════════════════════════════════════════════════
# RIGHT PANEL — band-edge shift schematic
# ════════════════════════════════════════════════════════════════════════════
ax = ax_sc
ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')

# Block geometry and energy levels
xl,  xr  = 0.6, 3.4
xl2, xr2 = 6.0, 8.8
Ec0, Ev0 = 6.8, 3.6       # unstrained edges
dEc, dEv = 0.70, 0.40     # shifts under tensile strain
Ec1, Ev1 = Ec0+dEc, Ev0-dEv
mid = (xr + xl2) / 2

def draw_block(x0, x1, Ec, Ev, ec_lbl, ev_lbl, bot_lbl):
    ax.fill_between([x0,x1], [0,0],   [Ev,Ev], color=C_VB, alpha=0.18)
    ax.fill_between([x0,x1], [Ec,Ec], [10,10], color=C_CB, alpha=0.18)
    ax.plot([x0,x1], [Ec,Ec], color=C_CB, lw=2.5)
    ax.plot([x0,x1], [Ev,Ev], color=C_VB, lw=2.5)
    cx = (x0+x1)/2
    ax.text(cx, Ec+0.30, ec_lbl, ha='center', fontsize=9,   color=C_CB)
    ax.text(cx, Ev-0.52, ev_lbl, ha='center', fontsize=9,   color=C_VB)
    ax.text(cx, (Ec+Ev)/2, r'$E_g$', ha='center', fontsize=9,
            color=C_GREY, style='italic')
    ax.text(cx, -0.65, bot_lbl, ha='center', fontsize=9.5,
            bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='gray', lw=0.8))

draw_block(xl,  xr,  Ec0, Ev0,
           r'$E_c^{(0)}$', r'$E_v^{(0)}$',
           r'$\varepsilon_h = 0$')
draw_block(xl2, xr2, Ec1, Ev1,
           r'$E_c^{(0)}+a_c\varepsilon_h$',
           r'$E_v^{(0)}-a_v\varepsilon_h$',
           r'$\varepsilon_h > 0\;$(tensile)')

# Dashed reference lines
for y, col in [(Ec0, C_CB), (Ev0, C_VB)]:
    ax.plot([xr, mid-0.05], [y, y], ls='--', color=col, lw=1.0, alpha=0.55)

# Gap arrows
def darrow(x, y1, y2, col, lw=1.6):
    ax.annotate('', xy=(x,y2), xytext=(x,y1),
                arrowprops=dict(arrowstyle='<->', color=col,
                                lw=lw, mutation_scale=14))

darrow(mid, Ev0, Ec0, C_GREY,  1.2)   # unstrained
darrow(mid, Ev1, Ec1, C_GREEN, 1.9)   # strained

ax.text(mid-0.28, (Ec0+Ev0)/2, r'$E_g^{(0)}$',
        fontsize=9, color=C_GREY,  ha='right', va='center')
ax.text(mid+0.25, (Ec1+Ev1)/2, r'$E_g(\varepsilon_h)$',
        fontsize=9, color=C_GREEN, ha='left',  va='center')

# Shift labels
rx = xr2 + 0.28
ax.text(rx, (Ec0+Ec1)/2, r'$\uparrow\;a_c\varepsilon_h$',
        fontsize=9, va='center', color=C_CB)
ax.text(rx, (Ev0+Ev1)/2, r'$\downarrow\;a_v\varepsilon_h$',
        fontsize=9, va='center', color=C_VB)

# Key equation
ax.text(5.0, 9.25,
        r'$\Delta E_g = (a_c + a_v)\,\varepsilon_h$',
        ha='center', fontsize=12, color=C_GREEN,
        bbox=dict(boxstyle='round,pad=0.4', fc='honeydew',
                  ec=C_GREEN, lw=1.5))

ax.set_title('Hydrostatic band-edge shift\n'
             r'(Hamaguchi §4.7.2)', fontsize=11)

# ════════════════════════════════════════════════════════════════════════════
# Save
# ════════════════════════════════════════════════════════════════════════════
fig.suptitle('Silicon electronic structure and strain-induced band-gap shift',
             fontsize=13, y=1.01)
fig.tight_layout()

OUT = '../paper/Images/fig_band_shift.png'
os.makedirs(os.path.dirname(OUT), exist_ok=True)
fig.savefig(OUT, dpi=200, bbox_inches='tight')
print(f'Figure saved → {OUT}')
plt.show()


Loaded real GPAW band structure from bs.json
Figure saved → ../paper/Images/fig_band_shift.png


/tmp/ipykernel_34001/520797976.py:213: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
